# 🟦 KuboERP · Taller de TDD, BDD y Calidad Ágil con Python

**AUTOMATIZA • INTEGRA • OPTIMIZA**

Solución del *Taller Integral de TDD, BDD y Calidad Ágil con Python en Google Colab* aplicada a los módulos de
**Compras, Logística y Almacén** de KuboERP, el ERP para una empresa que vende productos eléctricos y presta
servicios eléctricos residenciales, comerciales e industriales.

| | |
|---|---|
| **Materia** | Proyecto de Ingeniería de Software II |
| **Profesor** | Carlos Eduardo Mujica Reyes |
| **Institución** | Universidad de Cundinamarca |
| **Integrante** | Oscar Javier Bello Daza |

**Cómo usarlo:** en Colab, menú *Entorno de ejecución → Ejecutar todas*. Cada celda `%%writefile` crea un archivo del
proyecto y cada celda con `!` ejecuta las pruebas.

### Datos de KuboERP (los mismos de los mockups)
| Proveedor | NIT | | Código | Producto |
|---|---|---|---|---|
| Distribuidora Eléctrica del Centro | 900123456 | | CAB-THHN-12 | Cable THHN 12 AWG (rollo 100 m) |
| Importadora LuzLED | 901555777 | | BRK-2X30 | Breaker 2x30A |
| Procables S.A.S. | 900888111 | | LED-6060-48W | Panel LED 60x60 48W |
| Schneider Electric Colombia | — | | TOMA-DOBLE-PT | Toma doble con polo a tierra |

Consecutivos: solicitud **SC-0001**, cotizaciones **COT-0001**, órdenes de compra **OC-0001**.

## 0. Preparar el entorno
`pytest` ya viene en Colab; se instalan `behave` (BDD) y `coverage` (cobertura).

In [ ]:
!pip install -q behave coverage pytest

In [ ]:
import os, sys
os.environ["PYTHONIOENCODING"] = "utf-8"
for carpeta in ("kuboerp", "tests", "features/steps"):
    os.makedirs(carpeta, exist_ok=True)
print("Carpetas listas:", sorted(os.listdir(".")))

In [ ]:
%%writefile kuboerp/__init__.py
"""KuboERP · Automatiza • Integra • Optimiza.

Módulos de Compras, Logística y Almacén para una empresa de productos
y servicios eléctricos: proveedores, cotizaciones, órdenes de compra e inventario.
"""

In [ ]:
%%writefile kuboerp/excepciones.py
"""Excepciones de negocio de KuboERP (Compras, Logística y Almacén)."""


class InvalidSupplierException(Exception):
    """El proveedor no cumple las reglas de registro."""


class SupplierHasActiveOrdersException(Exception):
    """El proveedor tiene órdenes de compra activas y no se puede eliminar."""


class InvalidQuotationException(Exception):
    """La cotización no cumple las reglas de registro."""


class PurchaseOrderException(Exception):
    """La orden de compra no se puede aprobar."""


class InvalidProductException(Exception):
    """El producto no cumple las reglas del inventario."""


class InsufficientStockException(Exception):
    """El retiro supera el stock disponible."""

## 1. Ejercicio práctico 1 · TDD de `InventoryService.update_stock(product_id, quantity)`
Regla: si la cantidad es negativa (salida) y el stock resultante quedaría por debajo de cero, se lanza
`InsufficientStockException`. Se sigue el ciclo **🔴 Rojo → 🟢 Verde → 🔵 Refactor**.

### 🔴 Rojo · primero la prueba
El servicio todavía **no tiene** `update_stock`, así que las pruebas deben fallar.

In [ ]:
import unittest
from kuboerp.excepciones import InsufficientStockException

class InventoryService:                      # versión inicial: aún no existe update_stock
    def __init__(self):
        self.stock = {}
    def register_product(self, codigo, stock):
        self.stock[codigo] = stock

class TestUpdateStockTDD(unittest.TestCase):
    def setUp(self):
        self.service = InventoryService()
        self.service.register_product("BRK-2X30", 40)        # Breaker 2x30A
        self.service.register_product("TOMA-DOBLE-PT", 2)    # Toma doble con polo a tierra

    def test_salida_que_deja_stock_negativo_lanza_excepcion(self):
        with self.assertRaises(InsufficientStockException):
            self.service.update_stock("TOMA-DOBLE-PT", -5)
        self.assertEqual(self.service.stock["TOMA-DOBLE-PT"], 2)

    def test_salida_valida_resta_del_stock(self):
        self.assertEqual(self.service.update_stock("BRK-2X30", -5), 35)

unittest.main(argv=["ignorado", "TestUpdateStockTDD"], exit=False, verbosity=2)

Resultado esperado: **2 errores** (`AttributeError: ... has no attribute 'update_stock'`). Esa es la fase Roja.

### 🟢 Verde · el código mínimo para pasar

In [ ]:
class InventoryService(InventoryService):
    def update_stock(self, product_id, quantity):
        if self.stock[product_id] + quantity < 0:
            raise InsufficientStockException("Stock insuficiente")
        self.stock[product_id] += quantity
        return self.stock[product_id]

unittest.main(argv=["ignorado", "TestUpdateStockTDD"], exit=False, verbosity=2)

Resultado esperado: **OK**. Las dos pruebas pasan con la implementación mínima.

### 🔵 Refactor · mejorar el diseño sin romper las pruebas
En la versión final del módulo:
* la validación se extrae a `_validar_stock_resultante()`;
* el producto se guarda como `Product` (código, nombre, categoría y stock) en un repositorio;
* `retirar()` reutiliza `update_stock()` en lugar de repetir la regla;
* cada movimiento queda registrado para el kardex.

In [ ]:
%%writefile kuboerp/inventario.py
"""Taller 4 y Ejercicio 1 · Inventario de productos, materia prima e insumos eléctricos.

Versión final del ciclo TDD (después del Refactor): la validación del stock se
extrajo a _validar_stock_resultante() y retirar() reutiliza update_stock().
"""
from dataclasses import dataclass

from kuboerp.excepciones import InsufficientStockException, InvalidProductException

MSG_RETIRO_EXITOSO = "Retiro exitoso"
MSG_STOCK_INSUFICIENTE = "Stock insuficiente"


@dataclass
class Product:
    codigo: str
    nombre: str
    categoria: str
    stock: int = 0


class InMemoryProductRepository:
    def __init__(self):
        self._datos = {}
        self.movimientos = []

    def save(self, producto):
        self._datos[producto.codigo] = producto
        return producto

    def find_by_code(self, codigo):
        return self._datos.get(codigo)


class InventoryService:
    def __init__(self, repo=None):
        self.repo = repo if repo is not None else InMemoryProductRepository()

    def register_product(self, codigo, nombre, categoria, stock=0):
        if not codigo:
            raise InvalidProductException("El código es obligatorio")
        if not categoria:
            raise InvalidProductException("La categoría es obligatoria")
        if stock < 0:
            raise InvalidProductException("El stock inicial no puede ser negativo")
        if self.repo.find_by_code(codigo) is not None:
            raise InvalidProductException("El código ya está registrado")
        return self.repo.save(Product(codigo, nombre, categoria, stock))

    def update_stock(self, product_id, quantity):
        """Suma (entrada) o resta (salida, cantidad negativa) unidades al stock."""
        producto = self._buscar(product_id)
        nuevo_stock = producto.stock + quantity
        self._validar_stock_resultante(nuevo_stock)
        producto.stock = nuevo_stock
        self.repo.save(producto)
        self.repo.movimientos.append((product_id, quantity))
        return producto.stock

    def retirar(self, codigo, cantidad):
        self.update_stock(codigo, -cantidad)
        return MSG_RETIRO_EXITOSO

    def stock_de(self, codigo):
        return self._buscar(codigo).stock

    def _buscar(self, codigo):
        producto = self.repo.find_by_code(codigo)
        if producto is None:
            raise InvalidProductException("Producto no encontrado")
        return producto

    @staticmethod
    def _validar_stock_resultante(nuevo_stock):
        if nuevo_stock < 0:
            raise InsufficientStockException(MSG_STOCK_INSUFICIENTE)

In [ ]:
%%writefile tests/test_ep1_inventario_tdd.py
"""Ejercicio práctico 1 · TDD de InventoryService.update_stock(product_id, quantity)."""
import unittest

from kuboerp.excepciones import InsufficientStockException
from kuboerp.inventario import InventoryService


class TestUpdateStock(unittest.TestCase):
    def setUp(self):
        self.service = InventoryService()
        self.service.register_product("BRK-2X30", "Breaker 2x30A", "Breakers y tableros", 40)
        self.service.register_product("TOMA-DOBLE-PT", "Toma doble con polo a tierra", "Interruptores y tomas", 2)

    def test_salida_que_deja_stock_negativo_lanza_excepcion(self):
        with self.assertRaises(InsufficientStockException):
            self.service.update_stock("TOMA-DOBLE-PT", -5)
        self.assertEqual(self.service.stock_de("TOMA-DOBLE-PT"), 2)

    def test_salida_valida_resta_del_stock(self):
        self.assertEqual(self.service.update_stock("BRK-2X30", -5), 35)

    def test_entrada_suma_al_stock(self):
        self.assertEqual(self.service.update_stock("BRK-2X30", 10), 50)


if __name__ == "__main__":
    unittest.main()

In [ ]:
!{sys.executable} -m pytest tests/test_ep1_inventario_tdd.py -v

## 2. Ejercicio práctico 2 · `SupplierService` con `unittest` en una celda
Enunciado: `create_supplier(nit, nombre, correo)` lanza `InvalidSupplierException` si el NIT tiene menos de 9
caracteres o si el correo no contiene `@`. Tres pruebas: NIT válido, NIT inválido y correo inválido.

In [ ]:
import unittest

class InvalidSupplierException(Exception):
    pass

class SupplierService:
    def create_supplier(self, nit, nombre, correo):
        if nit is None or len(nit) < 9:
            raise InvalidSupplierException("NIT inválido")
        if "@" not in correo:
            raise InvalidSupplierException("Formato de correo inválido")
        return {"nit": nit, "nombre": nombre, "correo": correo, "estado": "ACTIVO"}

class TestSupplierServiceCelda(unittest.TestCase):
    def setUp(self):
        self.service = SupplierService()

    def test_nit_valido(self):
        proveedor = self.service.create_supplier("900123456", "Distribuidora Eléctrica del Centro",
                                                 "compras@electrocentro.com.co")
        self.assertEqual(proveedor["estado"], "ACTIVO")

    def test_nit_invalido(self):
        with self.assertRaises(InvalidSupplierException):
            self.service.create_supplier("123", "Procables S.A.S.", "ventas@procables.com.co")

    def test_correo_invalido(self):
        with self.assertRaises(InvalidSupplierException):
            self.service.create_supplier("901555777", "Importadora LuzLED", "ventas.luzled.co")

unittest.main(argv=["ignorado", "TestSupplierServiceCelda"], exit=False, verbosity=2)

Resultado esperado: `Ran 3 tests ... OK`.

En el proyecto, el servicio de proveedores completo (TDD aplicado: se extrajeron `es_nit_valido()` y
`es_correo_valido()`) valida además NIT numérico de 9 o 10 dígitos, nombre obligatorio y NIT único, y hace la
eliminación lógica del Taller 1.

In [ ]:
%%writefile kuboerp/proveedores.py
"""Taller 1 · CRUD de proveedores de material eléctrico."""
import re
from dataclasses import dataclass

from kuboerp.excepciones import InvalidSupplierException, SupplierHasActiveOrdersException

MSG_REGISTRADO = "Proveedor registrado exitosamente"
MSG_DESACTIVADO = "Proveedor desactivado"

_NIT = re.compile(r"^\d{9,10}$")
_CORREO = re.compile(r"^[\w.+-]+@[\w-]+(\.[\w-]+)+$")


@dataclass
class Supplier:
    nit: str
    nombre: str
    correo: str
    estado: str = "ACTIVO"
    presupuesto: int = 0


class InMemorySupplierRepository:
    def __init__(self):
        self._datos = {}

    def save(self, proveedor):
        self._datos[proveedor.nit] = proveedor
        return proveedor

    def find_by_nit(self, nit):
        return self._datos.get(nit)

    def count(self):
        return len(self._datos)


class InMemoryOrderRepository:
    """Repositorio mínimo de órdenes: solo responde si un proveedor tiene órdenes activas."""

    def __init__(self, nits_con_ordenes=()):
        self._nits = set(nits_con_ordenes)

    def has_active_orders(self, nit):
        return nit in self._nits


class SupplierService:
    def __init__(self, repo=None, repo_ordenes=None):
        self.repo = repo if repo is not None else InMemorySupplierRepository()
        self.repo_ordenes = repo_ordenes if repo_ordenes is not None else InMemoryOrderRepository()

    @staticmethod
    def es_nit_valido(nit):
        return bool(nit) and bool(_NIT.match(nit))

    @staticmethod
    def es_correo_valido(correo):
        return bool(correo) and bool(_CORREO.match(correo))

    def existe_por_nit(self, nit):
        return self.repo.find_by_nit(nit) is not None

    def create_supplier(self, nit, nombre, correo):
        if not self.es_nit_valido(nit):
            raise InvalidSupplierException("NIT inválido")
        if not nombre or not nombre.strip():
            raise InvalidSupplierException("El nombre es obligatorio")
        if not self.es_correo_valido(correo):
            raise InvalidSupplierException("Formato de correo inválido")
        if self.existe_por_nit(nit):
            raise InvalidSupplierException("El NIT ya está registrado")
        return self.repo.save(Supplier(nit, nombre.strip(), correo))

    def buscar(self, nit):
        proveedor = self.repo.find_by_nit(nit)
        if proveedor is None:
            raise InvalidSupplierException("Proveedor no encontrado")
        return proveedor

    def delete_supplier(self, nit):
        """Eliminación lógica: el proveedor pasa a INACTIVO si no tiene órdenes activas."""
        if self.repo_ordenes.has_active_orders(nit):
            raise SupplierHasActiveOrdersException(
                "No se puede eliminar un proveedor con órdenes de compra activas")
        proveedor = self.buscar(nit)
        proveedor.estado = "INACTIVO"
        self.repo.save(proveedor)
        return MSG_DESACTIVADO

In [ ]:
%%writefile tests/test_ep2_proveedores_unittest.py
"""Ejercicio práctico 2 · Tres pruebas unittest de SupplierService.create_supplier()."""
import unittest

from kuboerp.excepciones import InvalidSupplierException
from kuboerp.proveedores import SupplierService


class TestCreateSupplier(unittest.TestCase):
    def setUp(self):
        self.service = SupplierService()

    def test_nit_valido_crea_proveedor_activo(self):
        proveedor = self.service.create_supplier(
            "900123456", "Distribuidora Eléctrica del Centro", "compras@electrocentro.com.co")
        self.assertEqual(proveedor.estado, "ACTIVO")

    def test_nit_con_menos_de_9_caracteres_es_rechazado(self):
        with self.assertRaises(InvalidSupplierException):
            self.service.create_supplier("123", "Procables S.A.S.", "ventas@procables.com.co")

    def test_correo_sin_arroba_es_rechazado(self):
        with self.assertRaises(InvalidSupplierException):
            self.service.create_supplier("901555777", "Importadora LuzLED", "ventas.luzled.co")


if __name__ == "__main__":
    unittest.main()

In [ ]:
!{sys.executable} -m pytest tests/test_ep2_proveedores_unittest.py -v

## 3. Ejercicio práctico 3 · BDD con Behave: registro de una cotización
Escenario con un proveedor existente, una solicitud de cotización **ABIERTA**, el monto cotizado y la verificación
de que la cotización queda en estado **RECIBIDA**.

Primero el código de producción de cotizaciones y órdenes de compra (el entorno de Behave los usa todos).

In [ ]:
%%writefile kuboerp/cotizaciones.py
"""Taller 2 · Solicitudes y cotizaciones de proveedores."""
from dataclasses import dataclass

from kuboerp.excepciones import InvalidQuotationException

MSG_REGISTRADA = "Cotización registrada"


@dataclass
class Quotation:
    codigo: str
    solicitud: str
    nit_proveedor: str
    monto: int
    estado: str = "RECIBIDA"


class InMemoryQuotationRepository:
    def __init__(self):
        self.solicitudes = {}      # código → estado ("ABIERTA", "CERRADA")
        self.cotizaciones = []

    def add_request(self, codigo, estado="ABIERTA"):
        self.solicitudes[codigo] = estado

    def find_request(self, codigo):
        return self.solicitudes.get(codigo)

    def exists_by_request_and_supplier(self, solicitud, nit):
        return any(c.solicitud == solicitud and c.nit_proveedor == nit for c in self.cotizaciones)

    def save(self, cotizacion):
        self.cotizaciones.append(cotizacion)
        return cotizacion


class QuotationService:
    def __init__(self, repo=None):
        self.repo = repo if repo is not None else InMemoryQuotationRepository()

    def registrar(self, codigo, solicitud, nit_proveedor, monto):
        if self.repo.find_request(solicitud) != "ABIERTA":
            raise InvalidQuotationException("La solicitud no está abierta")
        if monto <= 0:
            raise InvalidQuotationException("Monto inválido")
        if self.repo.exists_by_request_and_supplier(solicitud, nit_proveedor):
            raise InvalidQuotationException("El proveedor ya cotizó esta solicitud")
        return self.repo.save(Quotation(codigo, solicitud, nit_proveedor, monto))

In [ ]:
%%writefile kuboerp/ordenes.py
"""Taller 3 · Aprobación de órdenes de compra contra el presupuesto del proveedor."""
from dataclasses import dataclass

from kuboerp.excepciones import PurchaseOrderException


@dataclass
class PurchaseOrder:
    codigo: str
    nit_proveedor: str
    monto: int
    estado: str = "PENDIENTE"


class InMemoryPurchaseOrderRepository:
    def __init__(self):
        self._datos = {}

    def save(self, orden):
        self._datos[orden.codigo] = orden
        return orden

    def find_by_codigo(self, codigo):
        return self._datos.get(codigo)

    def has_active_orders(self, nit):
        return any(o.nit_proveedor == nit and o.estado in ("PENDIENTE", "APROBADA")
                   for o in self._datos.values())


class PurchaseOrderService:
    def __init__(self, repo_proveedores, repo_ordenes):
        self.repo_proveedores = repo_proveedores
        self.repo_ordenes = repo_ordenes

    def approve(self, codigo):
        orden = self.repo_ordenes.find_by_codigo(codigo)
        if orden is None:
            raise PurchaseOrderException("Orden no encontrada")
        if orden.estado == "APROBADA":
            raise PurchaseOrderException("La orden ya está aprobada")
        proveedor = self.repo_proveedores.find_by_nit(orden.nit_proveedor)
        if orden.monto > proveedor.presupuesto:
            raise PurchaseOrderException("Presupuesto insuficiente")
        proveedor.presupuesto -= orden.monto
        self.repo_proveedores.save(proveedor)
        orden.estado = "APROBADA"
        return self.repo_ordenes.save(orden)

`environment.py` crea servicios nuevos antes de cada escenario y `context` comparte el estado entre pasos.

In [ ]:
%%writefile features/environment.py
"""Behave · prepara servicios nuevos (en memoria) antes de cada escenario."""
from kuboerp.cotizaciones import InMemoryQuotationRepository, QuotationService
from kuboerp.inventario import InventoryService
from kuboerp.ordenes import InMemoryPurchaseOrderRepository, PurchaseOrderService
from kuboerp.proveedores import InMemorySupplierRepository, SupplierService


def before_scenario(context, scenario):
    context.repo_proveedores = InMemorySupplierRepository()
    context.repo_ordenes = InMemoryPurchaseOrderRepository()
    context.repo_cotizaciones = InMemoryQuotationRepository()

    context.proveedores = SupplierService(context.repo_proveedores, context.repo_ordenes)
    context.ordenes = PurchaseOrderService(context.repo_proveedores, context.repo_ordenes)
    context.cotizaciones = QuotationService(context.repo_cotizaciones)
    context.inventario = InventoryService()

    context.mensaje = None
    context.excepcion = None

    def ejecutar(accion):
        """Ejecuta el paso @when y guarda el mensaje de éxito o el de la excepción."""
        try:
            context.mensaje = accion()
            context.excepcion = None
        except Exception as error:  # noqa: BLE001 - el escenario verifica el mensaje
            context.excepcion = error
            context.mensaje = str(error)

    context.ejecutar = ejecutar

In [ ]:
%%writefile features/steps/mensajes_steps.py
"""Pasos comunes: verifican el mensaje que muestra KuboERP."""
from behave import then


@then('el sistema muestra "{mensaje}"')
def step_sistema_muestra(context, mensaje):
    assert context.mensaje == mensaje, f"Se esperaba '{mensaje}' y se obtuvo '{context.mensaje}'"


@then('el sistema responde "{mensaje}"')
def step_sistema_responde(context, mensaje):
    assert context.mensaje == mensaje, f"Se esperaba '{mensaje}' y se obtuvo '{context.mensaje}'"


@then('el sistema confirma "{mensaje}"')
def step_sistema_confirma(context, mensaje):
    assert context.mensaje == mensaje, f"Se esperaba '{mensaje}' y se obtuvo '{context.mensaje}'"

In [ ]:
%%writefile features/cotizaciones.feature
# language: es
# Ejercicio práctico 3 y Taller 2 · Cotizaciones de proveedores (Mockup 3 · Compras)
Característica: Registro de cotizaciones de material eléctrico
  Como jefe de compras
  Quiero registrar las cotizaciones que envían los proveedores
  Para comparar precios antes de generar la orden de compra

  Antecedentes:
    Dado que existe el proveedor "Importadora LuzLED" con NIT "901555777"
    Y que existe una solicitud "SC-0001" en estado "ABIERTA"

  Escenario: Registrar la cotización de un proveedor
    Cuando el proveedor con NIT "901555777" cotiza la solicitud "SC-0001" por 2300000 con el código "COT-0001"
    Entonces la cotización "COT-0001" queda en estado "RECIBIDA"
    Y el sistema responde "Cotización registrada"

  Esquema del escenario: Validar monto de cotización
    Cuando registro una cotización por <monto>
    Entonces el sistema responde "<resultado>"

    Ejemplos:
      | monto   | resultado             |
      | 2500000 | Cotización registrada |
      | 0       | Monto inválido        |
      | -1000   | Monto inválido        |

  Escenario: Rechazar una segunda cotización del mismo proveedor
    Dado que el proveedor con NIT "901555777" ya cotizó la solicitud "SC-0001"
    Cuando registro una cotización por 2500000
    Entonces el sistema responde "El proveedor ya cotizó esta solicitud"

In [ ]:
%%writefile features/steps/cotizaciones_steps.py
"""Ejercicio práctico 3 y Taller 2 · Step definitions de cotizaciones."""
from behave import given, then, when

from kuboerp.cotizaciones import MSG_REGISTRADA


@given('que existe el proveedor "{nombre}" con NIT "{nit}"')
def step_existe_proveedor(context, nombre, nit):
    context.proveedores.create_supplier(nit, nombre, "ventas@importadoraluzled.com")
    context.nit = nit


@given('que existe una solicitud "{codigo}" en estado "{estado}"')
def step_existe_solicitud(context, codigo, estado):
    context.repo_cotizaciones.add_request(codigo, estado)
    context.solicitud = codigo


@when('el proveedor con NIT "{nit}" cotiza la solicitud "{solicitud}" por {monto:d} con el código "{codigo}"')
def step_cotiza(context, nit, solicitud, monto, codigo):
    def accion():
        context.cotizacion = context.cotizaciones.registrar(codigo, solicitud, nit, monto)
        return MSG_REGISTRADA
    context.ejecutar(accion)


@then('la cotización "{codigo}" queda en estado "{estado}"')
def step_cotizacion_en_estado(context, codigo, estado):
    assert context.cotizacion.codigo == codigo
    assert context.cotizacion.estado == estado


@when('registro una cotización por {monto:d}')
def step_registro_cotizacion(context, monto):
    def accion():
        context.cotizaciones.registrar("COT-0002", context.solicitud, context.nit, monto)
        return MSG_REGISTRADA
    context.ejecutar(accion)


@given('que el proveedor con NIT "{nit}" ya cotizó la solicitud "{solicitud}"')
def step_ya_cotizo(context, nit, solicitud):
    context.cotizaciones.registrar("COT-0001", solicitud, nit, 2300000)

In [ ]:
!{sys.executable} -m behave features/cotizaciones.feature --no-color

## 4. Talleres integradores por módulo CRUD
Secuencia aplicada: **escenario Gherkin → prueba unitaria (unittest/pytest) → conexión con Behave → cobertura**.

### Taller 1 · CRUD de Proveedores
Reglas: NIT único y válido, correo con formato correcto, nombre obligatorio, no se elimina un proveedor con
órdenes activas. **Ejercicio:** la prueba del caso en que el proveedor *sí* puede eliminarse
(`test_deberia_desactivar_proveedor_sin_ordenes_activas`). También incluye el ejemplo de *fixture +
parametrización + mock* de la sección 3.

In [ ]:
%%writefile tests/test_taller1_proveedores.py
"""Taller 1 · Eliminar (desactivar) proveedores, con el repositorio de órdenes simulado."""
import unittest
from unittest.mock import MagicMock

import pytest

from kuboerp.excepciones import SupplierHasActiveOrdersException
from kuboerp.proveedores import SupplierService


class TestEliminarProveedor(unittest.TestCase):
    def test_no_deberia_eliminar_con_ordenes_activas(self):
        repo_ordenes = MagicMock()
        repo_ordenes.has_active_orders.return_value = True
        service = SupplierService(repo_ordenes=repo_ordenes)
        service.create_supplier("900888111", "Procables S.A.S.", "ventas@procables.com.co")

        with self.assertRaises(SupplierHasActiveOrdersException):
            service.delete_supplier("900888111")
        self.assertEqual(service.buscar("900888111").estado, "ACTIVO")

    def test_deberia_desactivar_proveedor_sin_ordenes_activas(self):
        repo_ordenes = MagicMock()
        repo_ordenes.has_active_orders.return_value = False
        service = SupplierService(repo_ordenes=repo_ordenes)
        service.create_supplier("901555777", "Importadora LuzLED", "ventas@importadoraluzled.com")

        mensaje = service.delete_supplier("901555777")

        self.assertEqual(mensaje, "Proveedor desactivado")
        self.assertEqual(service.buscar("901555777").estado, "INACTIVO")
        repo_ordenes.has_active_orders.assert_called_once_with("901555777")


# Ejemplo de la sección 3: fixture + parametrización + mock con pytest
@pytest.fixture
def supplier_service():
    repo = MagicMock()
    return SupplierService(repo)


@pytest.mark.parametrize("nit,esperado", [
    ("900123456", True),
    ("123", False),
    ("", False),
])
def test_validar_nit(supplier_service, nit, esperado):
    assert supplier_service.es_nit_valido(nit) == esperado

In [ ]:
%%writefile features/proveedores.feature
# language: es
# Taller 1 · CRUD de Proveedores (Mockup 3 · Compras, Logística y Almacén)
Característica: Gestión de proveedores de material eléctrico en KuboERP
  Como jefe de compras de la empresa
  Quiero registrar proveedores con NIT y correo válidos
  Para mantener actualizada la base de fabricantes, mayoristas e importadores

  Escenario: Registrar un proveedor válido
    Dado que no existe un proveedor con NIT "900123456"
    Cuando registro el proveedor con NIT "900123456", nombre "Distribuidora Eléctrica del Centro" y correo "compras@electrocentro.com.co"
    Entonces el proveedor se crea con estado "ACTIVO"
    Y el sistema muestra "Proveedor registrado exitosamente"

  Escenario: Rechazar proveedor con NIT duplicado
    Dado que ya existe un proveedor con NIT "900123456"
    Cuando intento registrar otro proveedor con el mismo NIT
    Entonces el sistema muestra "El NIT ya está registrado"
    Y no se crea un nuevo proveedor

  Esquema del escenario: Validar formatos de correo
    Cuando registro un proveedor con correo "<correo>"
    Entonces el sistema responde "<resultado>"

    Ejemplos:
      | correo                       | resultado                  |
      | ventas@importadoraluzled.com | Correo válido              |
      | sin-arroba.com               | Formato de correo inválido |
      | usuario@                     | Formato de correo inválido |

  Escenario: Eliminar proveedor sin órdenes activas
    Dado que existe un proveedor "Importadora LuzLED" con NIT "901555777" sin órdenes de compra activas
    Cuando solicito eliminar el proveedor
    Entonces el proveedor cambia a estado "INACTIVO"
    Y el sistema confirma "Proveedor desactivado"

  Escenario: No eliminar proveedor con órdenes activas
    Dado que existe un proveedor "Procables S.A.S." con NIT "900888111" y la orden de compra "OC-0001" en estado "PENDIENTE"
    Cuando solicito eliminar el proveedor
    Entonces el sistema muestra "No se puede eliminar un proveedor con órdenes de compra activas"
    Y el proveedor sigue en estado "ACTIVO"

In [ ]:
%%writefile features/steps/proveedores_steps.py
"""Taller 1 · Step definitions de proveedores."""
from behave import given, then, when

from kuboerp.ordenes import PurchaseOrder
from kuboerp.proveedores import MSG_REGISTRADO


@given('que no existe un proveedor con NIT "{nit}"')
def step_no_existe_proveedor(context, nit):
    assert not context.proveedores.existe_por_nit(nit)


@when('registro el proveedor con NIT "{nit}", nombre "{nombre}" y correo "{correo}"')
def step_registro_proveedor(context, nit, nombre, correo):
    def accion():
        context.proveedor = context.proveedores.create_supplier(nit, nombre, correo)
        return MSG_REGISTRADO
    context.ejecutar(accion)


@then('el proveedor se crea con estado "{estado}"')
def step_proveedor_creado(context, estado):
    assert context.proveedor is not None
    assert context.proveedor.estado == estado


@given('que ya existe un proveedor con NIT "{nit}"')
def step_ya_existe_proveedor(context, nit):
    context.proveedores.create_supplier(nit, "Distribuidora Eléctrica del Centro", "compras@electrocentro.com.co")
    context.nit = nit
    context.proveedores_antes = context.repo_proveedores.count()


@when('intento registrar otro proveedor con el mismo NIT')
def step_registro_duplicado(context):
    context.ejecutar(lambda: context.proveedores.create_supplier(
        context.nit, "Schneider Electric Colombia", "ventas@schneider.com.co"))


@then('no se crea un nuevo proveedor')
def step_no_se_crea(context):
    assert context.repo_proveedores.count() == context.proveedores_antes


@when('registro un proveedor con correo "{correo}"')
def step_registro_con_correo(context, correo):
    def accion():
        context.proveedores.create_supplier("901555777", "Importadora LuzLED", correo)
        return "Correo válido"
    context.ejecutar(accion)


@given('que existe un proveedor "{nombre}" con NIT "{nit}" sin órdenes de compra activas')
def step_proveedor_sin_ordenes(context, nombre, nit):
    context.proveedores.create_supplier(nit, nombre, "contacto@proveedor.com.co")
    context.nit = nit
    assert not context.repo_ordenes.has_active_orders(nit)


@given('que existe un proveedor "{nombre}" con NIT "{nit}" y la orden de compra "{orden}" en estado "{estado}"')
def step_proveedor_con_orden(context, nombre, nit, orden, estado):
    context.proveedores.create_supplier(nit, nombre, "contacto@proveedor.com.co")
    context.repo_ordenes.save(PurchaseOrder(orden, nit, 3_000_000, estado))
    context.nit = nit


@when('solicito eliminar el proveedor')
def step_eliminar_proveedor(context):
    context.ejecutar(lambda: context.proveedores.delete_supplier(context.nit))


@then('el proveedor cambia a estado "{estado}"')
def step_proveedor_cambia_estado(context, estado):
    assert context.proveedores.buscar(context.nit).estado == estado


@then('el proveedor sigue en estado "{estado}"')
def step_proveedor_sigue_estado(context, estado):
    assert context.proveedores.buscar(context.nit).estado == estado

In [ ]:
!{sys.executable} -m pytest tests/test_taller1_proveedores.py -v
!{sys.executable} -m behave features/proveedores.feature --no-color

### Taller 2 · Solicitud de Cotizaciones
Reglas: solo solicitudes **ABIERTAS**, monto > 0 y sin cotizaciones duplicadas del mismo proveedor.
**Ejercicio:** `@pytest.mark.parametrize` con los tres montos; para montos ≤ 0 se verifica la excepción y que
**no** se llama a `save` del repositorio (simulado con `MagicMock`). El esquema Gherkin del taller ya se ejecutó en
`cotizaciones.feature` (sección 3).

In [ ]:
%%writefile tests/test_taller2_cotizaciones.py
"""Taller 2 · Validar el monto de una cotización con @pytest.mark.parametrize."""
from unittest.mock import MagicMock

import pytest

from kuboerp.cotizaciones import QuotationService
from kuboerp.excepciones import InvalidQuotationException


@pytest.fixture
def repo():
    repo = MagicMock()
    repo.find_request.return_value = "ABIERTA"
    repo.exists_by_request_and_supplier.return_value = False
    repo.save.side_effect = lambda cotizacion: cotizacion
    return repo


@pytest.mark.parametrize("monto,resultado", [
    (2500000, "Cotización registrada"),
    (0, "Monto inválido"),
    (-1000, "Monto inválido"),
], ids=["monto_2500000", "monto_0", "monto_negativo_1000"])
def test_validar_monto_de_cotizacion(repo, monto, resultado):
    service = QuotationService(repo)

    if monto > 0:
        cotizacion = service.registrar("COT-0001", "SC-0001", "901555777", monto)
        assert cotizacion.estado == "RECIBIDA"
        assert resultado == "Cotización registrada"
        repo.save.assert_called_once()
    else:
        with pytest.raises(InvalidQuotationException, match=resultado):
            service.registrar("COT-0001", "SC-0001", "901555777", monto)
        repo.save.assert_not_called()


def test_rechaza_cotizacion_sobre_solicitud_cerrada(repo):
    repo.find_request.return_value = "CERRADA"

    with pytest.raises(InvalidQuotationException, match="La solicitud no está abierta"):
        QuotationService(repo).registrar("COT-0002", "SC-0002", "901555777", 2300000)
    repo.save.assert_not_called()


def test_rechaza_segunda_cotizacion_del_mismo_proveedor(repo):
    repo.exists_by_request_and_supplier.return_value = True

    with pytest.raises(InvalidQuotationException, match="El proveedor ya cotizó esta solicitud"):
        QuotationService(repo).registrar("COT-0003", "SC-0001", "901555777", 2300000)
    repo.save.assert_not_called()

In [ ]:
!{sys.executable} -m pytest tests/test_taller2_cotizaciones.py -v

### Taller 3 · Aprobación de Órdenes de Compra
Reglas: la orden no puede exceder el presupuesto del proveedor, una orden aprobada no se vuelve a aprobar y la
aprobación descuenta el presupuesto. **Ejercicio:** el step definition `@then` consulta el presupuesto actualizado
en el repositorio y lo compara con el esperado. Un tipo propio `Pesos` convierte `$10.000.000` en número.

In [ ]:
%%writefile features/ordenes_compra.feature
# language: es
# Taller 3 · Aprobación de Órdenes de Compra (Mockup 3 · Compras)
Característica: Aprobación de órdenes de compra contra el presupuesto del proveedor
  Como jefe de compras
  Quiero aprobar las órdenes de compra sin exceder el presupuesto de cada proveedor
  Para controlar el gasto en material eléctrico

  Antecedentes:
    Dado que el proveedor "Distribuidora Eléctrica del Centro" tiene presupuesto disponible de $10.000.000

  Escenario: Aprobar orden dentro del presupuesto
    Dado que existe una orden "OC-0001" por $5.000.000 en estado "PENDIENTE"
    Cuando el jefe de compras aprueba la orden
    Entonces la orden cambia a "APROBADA"
    Y el presupuesto del proveedor se reduce a $5.000.000

  Escenario: Rechazar orden que excede el presupuesto
    Dado que existe una orden "OC-0002" por $15.000.000 en estado "PENDIENTE"
    Cuando el jefe de compras aprueba la orden
    Entonces el sistema muestra "Presupuesto insuficiente"
    Y la orden permanece en "PENDIENTE"
    Y el presupuesto disponible del proveedor sigue en $10.000.000

  Escenario: No volver a aprobar una orden ya aprobada
    Dado que existe una orden "OC-0003" por $1.000.000 en estado "APROBADA"
    Cuando el jefe de compras aprueba la orden
    Entonces el sistema muestra "La orden ya está aprobada"

In [ ]:
%%writefile features/steps/ordenes_steps.py
"""Taller 3 · Step definitions de la aprobación de órdenes de compra."""
import parse
from behave import given, register_type, then, when

from kuboerp.ordenes import PurchaseOrder
from kuboerp.proveedores import Supplier

NIT_PROVEEDOR = "900123456"


@parse.with_pattern(r"\$[\d.]+")
def parse_pesos(texto):
    """Convierte "$10.000.000" en 10000000."""
    return int(texto.replace("$", "").replace(".", ""))


register_type(Pesos=parse_pesos)


@given('que el proveedor "{nombre}" tiene presupuesto disponible de {presupuesto:Pesos}')
def step_presupuesto_proveedor(context, nombre, presupuesto):
    context.repo_proveedores.save(
        Supplier(NIT_PROVEEDOR, nombre, "compras@electrocentro.com.co", presupuesto=presupuesto))


@given('que existe una orden "{codigo}" por {monto:Pesos} en estado "{estado}"')
def step_existe_orden(context, codigo, monto, estado):
    context.repo_ordenes.save(PurchaseOrder(codigo, NIT_PROVEEDOR, monto, estado))
    context.codigo_orden = codigo


@when('el jefe de compras aprueba la orden')
def step_aprueba_orden(context):
    def accion():
        context.ordenes.approve(context.codigo_orden)
        return "Orden aprobada"
    context.ejecutar(accion)


@then('la orden cambia a "{estado}"')
def step_orden_cambia(context, estado):
    assert context.repo_ordenes.find_by_codigo(context.codigo_orden).estado == estado


@then('la orden permanece en "{estado}"')
def step_orden_permanece(context, estado):
    assert context.repo_ordenes.find_by_codigo(context.codigo_orden).estado == estado


@then('el presupuesto del proveedor se reduce a {esperado:Pesos}')
def step_presupuesto_reducido(context, esperado):
    # Se consulta el presupuesto actualizado en el repositorio y se compara con el esperado.
    actual = context.repo_proveedores.find_by_nit(NIT_PROVEEDOR).presupuesto
    assert actual == esperado, f"Presupuesto esperado {esperado}, actual {actual}"


@then('el presupuesto disponible del proveedor sigue en {esperado:Pesos}')
def step_presupuesto_sin_cambio(context, esperado):
    actual = context.repo_proveedores.find_by_nit(NIT_PROVEEDOR).presupuesto
    assert actual == esperado, f"Presupuesto esperado {esperado}, actual {actual}"

In [ ]:
%%writefile tests/test_taller3_ordenes.py
"""Taller 3 · Aprobación de órdenes de compra (pytest con fixtures)."""
import pytest

from kuboerp.excepciones import PurchaseOrderException
from kuboerp.ordenes import InMemoryPurchaseOrderRepository, PurchaseOrder, PurchaseOrderService
from kuboerp.proveedores import InMemorySupplierRepository, Supplier

NIT = "900123456"


@pytest.fixture
def repos():
    proveedores = InMemorySupplierRepository()
    proveedores.save(Supplier(NIT, "Distribuidora Eléctrica del Centro", "compras@electrocentro.com.co",
                              presupuesto=10_000_000))
    return proveedores, InMemoryPurchaseOrderRepository()


def test_aprobar_orden_dentro_del_presupuesto(repos):
    proveedores, ordenes = repos
    ordenes.save(PurchaseOrder("OC-0001", NIT, 5_000_000))

    orden = PurchaseOrderService(proveedores, ordenes).approve("OC-0001")

    assert orden.estado == "APROBADA"
    assert proveedores.find_by_nit(NIT).presupuesto == 5_000_000


def test_rechazar_orden_que_excede_el_presupuesto(repos):
    proveedores, ordenes = repos
    ordenes.save(PurchaseOrder("OC-0002", NIT, 15_000_000))

    with pytest.raises(PurchaseOrderException, match="Presupuesto insuficiente"):
        PurchaseOrderService(proveedores, ordenes).approve("OC-0002")
    assert ordenes.find_by_codigo("OC-0002").estado == "PENDIENTE"
    assert proveedores.find_by_nit(NIT).presupuesto == 10_000_000


def test_no_volver_a_aprobar_una_orden_aprobada(repos):
    proveedores, ordenes = repos
    ordenes.save(PurchaseOrder("OC-0003", NIT, 1_000_000, "APROBADA"))

    with pytest.raises(PurchaseOrderException, match="La orden ya está aprobada"):
        PurchaseOrderService(proveedores, ordenes).approve("OC-0003")

In [ ]:
!{sys.executable} -m behave features/ordenes_compra.feature --no-color
!{sys.executable} -m pytest tests/test_taller3_ordenes.py -v

### Taller 4 · Productos, materia prima e insumos en Inventario
Reglas: código único, stock inicial ≥ 0, categoría obligatoria, no se retira más del stock disponible.
**Ejercicio:** prueba con `subTest` que recorre los tres casos del esquema y verifica el stock final o la excepción.

In [ ]:
%%writefile features/inventario.feature
# language: es
# Taller 4 y Ejercicio práctico 1 · Inventario de la Bodega Principal (Mockup 3 · Almacén)
Característica: Entradas y retiros de material eléctrico
  Como bodeguero de la Bodega Principal
  Quiero registrar entradas y retiros de productos
  Para mantener el stock de seguridad de cables, breakers y luminarias

  Esquema del escenario: Retiro de inventario
    Dado que el producto "<codigo>" tiene stock <stock>
    Cuando se retiran <retiro> unidades
    Entonces el stock final es <final>
    Y el sistema responde "<mensaje>"

    Ejemplos:
      | codigo        | stock | retiro | final | mensaje            |
      | CAB-THHN-12   | 10    | 3      | 7     | Retiro exitoso     |
      | BRK-2X30      | 5     | 5      | 0     | Retiro exitoso     |
      | TOMA-DOBLE-PT | 2     | 5      | 2     | Stock insuficiente |

  Escenario: Registrar una entrada de mercancía
    Dado que el producto "BRK-2X30" tiene stock 40
    Cuando se registra una entrada de 10 unidades
    Entonces el stock final es 50

  Escenario: Rechazar una salida que deja el stock negativo
    Dado que el producto "TOMA-DOBLE-PT" tiene stock 2
    Cuando se retiran 5 unidades
    Entonces el sistema muestra "Stock insuficiente"
    Y el stock final es 2

In [ ]:
%%writefile features/steps/inventario_steps.py
"""Taller 4 · Step definitions de inventario."""
from behave import given, then, when

CATALOGO = {
    "CAB-THHN-12": ("Cable THHN 12 AWG (rollo 100 m)", "Cables"),
    "BRK-2X30": ("Breaker 2x30A", "Breakers y tableros"),
    "LED-6060-48W": ("Panel LED 60x60 48W", "Iluminación"),
    "TOMA-DOBLE-PT": ("Toma doble con polo a tierra", "Interruptores y tomas"),
}


@given('que el producto "{codigo}" tiene stock {stock:d}')
def step_producto_con_stock(context, codigo, stock):
    nombre, categoria = CATALOGO.get(codigo, (codigo, "Material eléctrico"))
    context.inventario.register_product(codigo, nombre, categoria, stock)
    context.codigo = codigo


@when('se retiran {cantidad:d} unidades')
def step_se_retiran(context, cantidad):
    context.ejecutar(lambda: context.inventario.retirar(context.codigo, cantidad))


@when('se registra una entrada de {cantidad:d} unidades')
def step_entrada(context, cantidad):
    context.ejecutar(lambda: context.inventario.update_stock(context.codigo, cantidad))


@then('el stock final es {final:d}')
def step_stock_final(context, final):
    assert context.inventario.stock_de(context.codigo) == final

In [ ]:
%%writefile tests/test_taller4_inventario.py
"""Taller 4 · Retiro de inventario recorriendo los casos del esquema con subTest."""
import unittest

from kuboerp.excepciones import InsufficientStockException, InvalidProductException
from kuboerp.inventario import InventoryService

CASOS = [
    # codigo,          stock, retiro, final, mensaje
    ("CAB-THHN-12",    10,    3,      7,     "Retiro exitoso"),
    ("BRK-2X30",       5,     5,      0,     "Retiro exitoso"),
    ("TOMA-DOBLE-PT",  2,     5,      2,     "Stock insuficiente"),
]


class TestRetiroInventario(unittest.TestCase):
    def test_retiro_de_inventario(self):
        for codigo, stock, retiro, final, mensaje in CASOS:
            with self.subTest(codigo=codigo, stock=stock, retiro=retiro):
                service = InventoryService()
                service.register_product(codigo, f"Producto {codigo}", "Material eléctrico", stock)

                if retiro > stock:
                    with self.assertRaises(InsufficientStockException) as error:
                        service.retirar(codigo, retiro)
                    self.assertEqual(str(error.exception), mensaje)
                else:
                    self.assertEqual(service.retirar(codigo, retiro), mensaje)
                self.assertEqual(service.stock_de(codigo), final)

    def test_no_registra_codigo_duplicado(self):
        service = InventoryService()
        service.register_product("BRK-2X30", "Breaker 2x30A", "Breakers y tableros", 40)
        with self.assertRaises(InvalidProductException):
            service.register_product("BRK-2X30", "Otro breaker", "Breakers y tableros", 5)

    def test_no_registra_sin_categoria_ni_con_stock_negativo(self):
        service = InventoryService()
        with self.assertRaises(InvalidProductException):
            service.register_product("LED-6060-48W", "Panel LED 60x60 48W", "", 10)
        with self.assertRaises(InvalidProductException):
            service.register_product("LED-6060-48W", "Panel LED 60x60 48W", "Iluminación", -1)


if __name__ == "__main__":
    unittest.main()

In [ ]:
!{sys.executable} -m behave features/inventario.feature --no-color
!{sys.executable} -m pytest tests/test_taller4_inventario.py -v

## 5. Ejercicio práctico 4 · Cobertura con `coverage.py`
### Paso 1: medir la cobertura actual e identificar líneas sin cubrir

In [ ]:
!{sys.executable} -m coverage run --source=kuboerp -m pytest -q
!{sys.executable} -m coverage report -m

La columna **Missing** muestra las líneas que ninguna prueba ejecuta. Por ejemplo, la línea 72 de `proveedores.py`
(regla *“El NIT ya está registrado”*) solo la probaba el escenario BDD, no una prueba unitaria; y el repositorio en
memoria de cotizaciones no se ejecutaba porque las pruebas del Taller 2 usan `MagicMock`.

### Paso 2: agregar pruebas para esas líneas y volver a medir

In [ ]:
%%writefile tests/test_ep4_cobertura.py
"""Ejercicio práctico 4 · Pruebas agregadas para cubrir las líneas que coverage reportó sin probar.

Antes de este archivo la cobertura era 92 %. Líneas faltantes:
  cotizaciones.py 20-21, 24, 27, 30, 33-34 · repositorio en memoria (las pruebas usaban MagicMock)
  inventario.py   41, 70                    · código vacío y producto inexistente
  ordenes.py      27, 39                    · órdenes activas en el repositorio y orden inexistente
  proveedores.py  35, 45, 68, 72, 78        · conteo, repositorio de órdenes, nombre vacío,
                                              NIT duplicado y proveedor inexistente
Hallazgo: la regla "NIT único" (línea 72) solo la probaba el escenario BDD de Behave;
ninguna prueba unitaria la cubría. Con este archivo la cobertura sube a 100 %.
"""
import pytest

from kuboerp.cotizaciones import InMemoryQuotationRepository, QuotationService
from kuboerp.excepciones import (InvalidProductException, InvalidQuotationException,
                                 InvalidSupplierException, PurchaseOrderException)
from kuboerp.inventario import InventoryService
from kuboerp.ordenes import InMemoryPurchaseOrderRepository, PurchaseOrder, PurchaseOrderService
from kuboerp.proveedores import InMemorySupplierRepository, SupplierService


def test_cotizaciones_con_repositorio_en_memoria():
    repo = InMemoryQuotationRepository()
    repo.add_request("SC-0001")
    service = QuotationService(repo)

    cotizacion = service.registrar("COT-0001", "SC-0001", "901555777", 2300000)

    assert cotizacion.estado == "RECIBIDA"
    with pytest.raises(InvalidQuotationException, match="El proveedor ya cotizó esta solicitud"):
        service.registrar("COT-0002", "SC-0001", "901555777", 2100000)
    with pytest.raises(InvalidQuotationException, match="La solicitud no está abierta"):
        service.registrar("COT-0003", "SC-9999", "901555777", 2100000)


def test_inventario_rechaza_codigo_vacio_y_producto_inexistente():
    service = InventoryService()
    with pytest.raises(InvalidProductException, match="El código es obligatorio"):
        service.register_product("", "Pinza voltiamperimétrica", "Herramientas", 8)
    with pytest.raises(InvalidProductException, match="Producto no encontrado"):
        service.stock_de("PINZA-VA-01")


def test_ordenes_inexistentes_y_ordenes_activas_por_proveedor():
    ordenes = InMemoryPurchaseOrderRepository()
    ordenes.save(PurchaseOrder("OC-0001", "900888111", 3_000_000))
    service = PurchaseOrderService(InMemorySupplierRepository(), ordenes)

    assert ordenes.has_active_orders("900888111")
    assert not ordenes.has_active_orders("901555777")
    with pytest.raises(PurchaseOrderException, match="Orden no encontrada"):
        service.approve("OC-9999")


def test_proveedores_nombre_vacio_inexistente_y_conteo():
    service = SupplierService()
    with pytest.raises(InvalidSupplierException, match="El nombre es obligatorio"):
        service.create_supplier("900123456", "   ", "compras@electrocentro.com.co")
    with pytest.raises(InvalidSupplierException, match="Proveedor no encontrado"):
        service.delete_supplier("999999999")

    service.create_supplier("900123456", "Distribuidora Eléctrica del Centro", "compras@electrocentro.com.co")
    assert service.repo.count() == 1


def test_proveedor_con_nit_duplicado_no_se_crea():
    service = SupplierService()
    service.create_supplier("900123456", "Distribuidora Eléctrica del Centro", "compras@electrocentro.com.co")

    with pytest.raises(InvalidSupplierException, match="El NIT ya está registrado"):
        service.create_supplier("900123456", "Schneider Electric Colombia", "ventas@schneider.com.co")
    assert service.repo.count() == 1

In [ ]:
!{sys.executable} -m coverage run --source=kuboerp -m pytest -q
!{sys.executable} -m coverage report -m
!{sys.executable} -m coverage xml

La cobertura sube y el archivo `coverage.xml` queda listo para SonarQube
(`sonar.python.coverage.reportPaths=coverage.xml`). Supera el umbral del **80 %** que pide la Definición de
Terminado (DoD).

## 6. Ejecución completa
Todas las pruebas unitarias (unittest + pytest) y todos los escenarios BDD (Behave).

In [ ]:
!{sys.executable} -m pytest -v

In [ ]:
!{sys.executable} -m behave --no-color -f progress

## 7. Cuestionario de autoevaluación · respuestas
| # | Pregunta | Respuesta correcta | En KuboERP |
|---|---|---|---|
| 1 | Orden del ciclo TDD | **Rojo → Verde → Refactorizar** | Ejercicio 1: `update_stock` falló, pasó y luego se refactorizó |
| 2 | Palabra clave de precondiciones en Gherkin | **Given (Dado)** | `Dado que el producto "BRK-2X30" tiene stock 40` |
| 3 | Biblioteca estándar de pruebas en Python | **unittest** | Ejercicios 1 y 2, Talleres 1 y 4 |
| 4 | Sintaxis de pytest en celdas de Colab | **ipytest** | Aquí se usó `%%writefile` + `pytest` (opción B) |
| 5 | Dónde van los step definitions de Behave | **En el directorio `steps/`** | `features/steps/*_steps.py` |
| 6 | Distribución de la pirámide de pruebas | **70 % unitarias, 20 % integración, 10 % E2E** | La mayoría de pruebas del proyecto son unitarias |
| 7 | Comando para medir cobertura | **`!coverage run -m pytest`** | Ejercicio 4 |
| 8 | Prueba unitaria adecuada para proveedores | **No crear un proveedor con NIT duplicado** | `test_proveedor_con_nit_duplicado_no_se_crea` |
| 9 | Diferencia entre TDD y BDD | **BDD: comportamiento de negocio; TDD: unidad de código** | `.feature` en español vs. `tests/` |
| 10 | Escenario de stock insuficiente | **Stock 2, retiro 5 → “Stock insuficiente” y el stock queda en 2** | `TOMA-DOBLE-PT` en el Taller 4 |